In [2]:
import sympy as sp


print("开始建立机械臂动力学模型...")

# 1. 定义符号
# q1: yaw, q2: pitch1, q3: pitch2
q1, q2, q3 = sp.symbols('q1 q2 q3', real=True)
dq1, dq2, dq3 = sp.symbols('dq1 dq2 dq3', real=True)
ddq1, ddq2, ddq3 = sp.symbols('ddq1 ddq2 ddq3', real=True)

# 连杆质量与尺寸参数
m1, m2 = sp.symbols('m1 m2', real=True, positive=True)
L1 = sp.symbols('L1', real=True, positive=True) # 第一连杆长度
d1, d2 = sp.symbols('d1 d2', real=True, positive=True) # 质心距离

# 转动惯量 (假设在质心主轴上的对角阵)
I1xx, I1yy, I1zz = sp.symbols('I1xx I1yy I1zz', real=True, positive=True)
I2xx, I2yy, I2zz = sp.symbols('I2xx I2yy I2zz', real=True, positive=True)

# 动态等效重力向量 (用于补偿基座的三维平动加速度, g_app = g_world - a_base)
gx, gy, gz = sp.symbols('gx gy gz', real=True)
g_vec = sp.Matrix([gx, gy, gz])

# 状态向量
q = sp.Matrix([q1, q2, q3])
dq = sp.Matrix([dq1, dq2, dq3])

print("正在计算正运动学...")
# 2. 坐标系与旋转矩阵
# 基坐标系：X向前，Y向左，Z向上
# 偏航关节 (Yaw, q1) 绕Z轴旋转
R_z = sp.Matrix([
    [sp.cos(q1), -sp.sin(q1), 0],
    [sp.sin(q1),  sp.cos(q1), 0],
    [0,           0,          1]
])

# 俯仰关节 (Pitch)。零位均指向+X方向。
# 规定：q为正时，机械臂抬起 (偏向+Z方向)。
# 根据右手法则，平面向Z轴正向旋转对应绕基坐标系负Y轴 (-Y) 的旋转。
def rot_y_neg(th):
    return sp.Matrix([
        [sp.cos(th), 0, -sp.sin(th)],
        [0,          1,  0         ],
        [sp.sin(th), 0,  sp.cos(th)]
    ])

R_p1 = rot_y_neg(q2)
R_p2 = rot_y_neg(q3)
R_p12 = rot_y_neg(q2 + q3)

# 连杆1与连杆2的全局旋转矩阵
R1 = R_z * R_p1
R2 = R_z * R_p12

# 质心位置 (Center of Mass)
# p_joint12 = [0,0,0]
p_c1 = R1 * sp.Matrix([d1, 0, 0])
p_joint3 = R1 * sp.Matrix([L1, 0, 0])
p_c2 = p_joint3 + R2 * sp.Matrix([d2, 0, 0])

print("正在计算平动与转动速度...")
# 3. 速度与动能
# 平动求导
v_c1 = p_c1.jacobian(q) * dq
v_c2 = p_c2.jacobian(q) * dq

# 角速度
# 偏航轴方向永远是 [0, 0, 1]^T
# 俯仰轴方向是绕Z轴yaw后的 -Y轴，即 R_z * [0, -1, 0]^T
omega1_global = sp.Matrix([0, 0, dq1]) + R_z * sp.Matrix([0, -dq2, 0])
omega2_global = sp.Matrix([0, 0, dq1]) + R_z * sp.Matrix([0, -dq2 - dq3, 0])

# 将角速度转换到各自连杆的局部坐标系下，以使用对角化的惯量矩阵
omega1_local = R1.T * omega1_global
omega2_local = R2.T * omega2_global

I1 = sp.diag(I1xx, I1yy, I1zz)
I2 = sp.diag(I2xx, I2yy, I2zz)

# 动能 (平动 + 转动)
T_trans = 0.5 * m1 * v_c1.dot(v_c1) + 0.5 * m2 * v_c2.dot(v_c2)
T_rot = 0.5 * (omega1_local.T * I1 * omega1_local)[0] + 0.5 * (omega2_local.T * I2 * omega2_local)[0]
T = sp.simplify(T_trans + T_rot)

# 势能
# U = - m * g_vec^T * p_c (g_vec 指向重力方向, 如 [0,0,-g]^T)
print("正在计算势能...")
V = - m1 * g_vec.dot(p_c1) - m2 * g_vec.dot(p_c2)
V = sp.simplify(V)

print("正在推导质量矩阵 M...")
# 4. 提取矩阵 M(q)
M = sp.zeros(3, 3)
for i in range(3):
    for j in range(3):
        # 动能对速度的二阶偏导为 M_ij
        M[i, j] = sp.diff(sp.diff(T, dq[i]), dq[j])
M = sp.simplify(M)

print("正在推导科里奥利矩阵 C...")
# 5. 提取科里奥利与离心力矩阵 C(q, dq) 使用 Christoffel symbol 
C = sp.zeros(3, 3)
for k in range(3):
    for j in range(3):
        c_val = 0
        for i in range(3):
            # c_ijk = 1/2 * ( dM_kj/dqi + dM_ki/dqj - dM_ij/dqk )
            c_ijk = 0.5 * (sp.diff(M[k, j], q[i]) + sp.diff(M[k, i], q[j]) - sp.diff(M[i, j], q[k]))
            c_val += c_ijk * dq[i]
        C[k, j] = c_val
C = sp.simplify(C)

print("正在推导重力项 G...")
# 6. 计算重力项 G(q)
G = sp.zeros(3, 1)
for i in range(3):
    G[i] = sp.diff(V, q[i])
G = sp.simplify(G)

# # ==== 打印结果 ====
# print("\n\n======== 动力学推导结果 ========")
# print("\n-> 质量矩阵 M(q) (3x3):")
# print(M)
# # sp.pprint(M)

# print("\n-> 科里奥利与离心力矩阵 C(q, \dot{q}) (3x3):")
# sp.pprint(C)

# print("\n-> 重力向量 g(q) (3x1):")
# sp.pprint(G)

# print("\n提示: 已化简至最简形式。如果需要将其转化为C++或Python控制代码，您可以使用 sp.ccode(M) 等生成代码。")

# sympy 的 ccode 不能直接打印 Matrix，需逐元素生成 C 代码
for i in range(G.rows):
  print(sp.ccode(G[i, 0], assign_to=f"G[{i}]"))

开始建立机械臂动力学模型...
正在计算正运动学...
正在计算平动与转动速度...
正在计算势能...
正在推导质量矩阵 M...
正在推导科里奥利矩阵 C...
正在推导重力项 G...
G[0] = (gx*sin(q1) - gy*cos(q1))*(d1*m1*cos(q2) + m2*(L1*cos(q2) + d2*cos(q2 + q3)));
G[1] = d1*m1*(gx*sin(q2)*cos(q1) + gy*sin(q1)*sin(q2) - gz*cos(q2)) + m2*(gx*(L1*sin(q2) + d2*sin(q2 + q3))*cos(q1) + gy*(L1*sin(q2) + d2*sin(q2 + q3))*sin(q1) - gz*(L1*cos(q2) + d2*cos(q2 + q3)));
G[2] = d2*m2*(gx*sin(q2 + q3)*cos(q1) + gy*sin(q1)*sin(q2 + q3) - gz*cos(q2 + q3));


In [3]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

print("=== 解析推导重力回归矩阵 Y(q, g_vec) ===")
# 1. 基于第一格的符号解，提取回归矩阵 Y
# 设需要辨识的组合参数为 b1 = m1*d1 + m2*L1, b2 = m2*d2
b1, b2 = sp.symbols('b1 b2', real=True)
theta_sym = sp.Matrix([b1, b2])

Y_sym = sp.zeros(3, 2)
# G 对 (m1*d1) 和 (m2*L1) 呈绝对线性关系，并且系数完全相同
# 直取 G 对 d1 求导除以 m1 提取出 b1 的系数矩阵列
Y_sym[:, 0] = sp.simplify(sp.diff(G, d1) / m1)
Y_sym[:, 1] = sp.simplify(sp.diff(G, d2) / m2)

print("\n-> 参数向量 theta:")
sp.pprint(theta_sym.T)
print("\n-> 回归矩阵 Y_sym(3x2):")
sp.pprint(Y_sym)

# 验证提取是否正确：Y*theta 即为重排各项组合后的 G
error = sp.simplify((Y_sym * sp.Matrix([m1*d1 + m2*L1, m2*d2])) - G)
print(f"\n-> 验证 Y*theta == G: {error.is_zero_matrix}")


print("\n\n=== 构建数值辨识算法 ===")
Y_func = sp.lambdify((q1, q2, q3, gx, gy, gz), Y_sym, "numpy")

def gravity_regressor_3d(q_val, g_val):
    """
    计算数值化的回归矩阵 Y
    q_val: [q1, q2, q3]
    g_val: [gx, gy, gz] 
    """
    res = np.array(Y_func(*q_val, *g_val), dtype=float)
    return res.reshape(3, 2)

def identify_gravity_params_3d(q_data, g_data, tau_g_data):
    N = len(q_data)
    Y_stack = np.vstack([gravity_regressor_3d(q_data[i], g_data[i]) for i in range(N)])
    tau_stack = tau_g_data.reshape(-1)
    
    reg = LinearRegression(fit_intercept=False)
    reg.fit(Y_stack, tau_stack)
    
    theta_hat = reg.coef_
    score = reg.score(Y_stack, tau_stack)
    return theta_hat, score


print("\n=== 使用实测数据进行参数辨识 ===")
# 读取采集的数据
df = pd.read_csv("data_20260503_074213.csv")
print(f"共 {len(df)} 组真实数据\n")

# 提取关节角度和力矩
q1_arr = df["q1"].values
q2_arr = df["q2"].values
q3_arr = df["q3"].values
q_data = np.column_stack((q1_arr, q2_arr, q3_arr))

tau_g_data = df[["tau1", "tau2", "tau3"]].values

# 构建固定重力场输入 (在此退化为静态重力场验证模型正确性)
g0 = 9.8 
g_data = np.zeros((len(df), 3))
g_data[:, 2] = -g0  # 设置 Z 轴重力，方向向下

# 执行辨识
theta_hat, r2 = identify_gravity_params_3d(q_data, g_data, tau_g_data)

# 与理论值对比
# 注意：在我们这套模型中，g 已经被吸收到输入变量 g_vec 中，Y_sym 内部会自行乘上 g。
# 因此我们辨识出来的 theta_hat 直接就是纯质量长度参数 (单位 kg·m)，不再包含重力加速度倍数。
m2_old, m3_old = 1.512, 1.7
l2_old = 0.29781
r2_len, r3_len = l2_old / 2, 0.3
b1_nominal = (m2_old * r2_len + m3_old * l2_old)
b2_nominal = m3_old * r3_len

print(f"辨识得出参数:   b1 = {theta_hat[0]:.4f} kg·m,  b2 = {theta_hat[1]:.4f} kg·m")
print(f"理论标称参数:   b1 = {b1_nominal:.4f} kg·m,  b2 = {b2_nominal:.4f} kg·m")
print(f"回归 R² 得分:   {r2:.6f}")
print(f"参数绝对误差:   Δb1 = {abs(theta_hat[0] - b1_nominal):.4f},  Δb2 = {abs(theta_hat[1] - b2_nominal):.4f}")

=== 解析推导重力回归矩阵 Y(q, g_vec) ===

-> 参数向量 theta:
[b₁  b₂]

-> 回归矩阵 Y_sym(3x2):
⎡         (gx⋅sin(q₁) - gy⋅cos(q₁))⋅cos(q₂)                          (gx⋅sin(q ↪
⎢                                                                              ↪
⎢gx⋅sin(q₂)⋅cos(q₁) + gy⋅sin(q₁)⋅sin(q₂) - gz⋅cos(q₂)  gx⋅sin(q₂ + q₃)⋅cos(q₁) ↪
⎢                                                                              ↪
⎣                         0                            gx⋅sin(q₂ + q₃)⋅cos(q₁) ↪

↪ ₁) - gy⋅cos(q₁))⋅cos(q₂ + q₃)               ⎤
↪                                             ⎥
↪  + gy⋅sin(q₁)⋅sin(q₂ + q₃) - gz⋅cos(q₂ + q₃)⎥
↪                                             ⎥
↪  + gy⋅sin(q₁)⋅sin(q₂ + q₃) - gz⋅cos(q₂ + q₃)⎦

-> 验证 Y*theta == G: True


=== 构建数值辨识算法 ===

=== 使用实测数据进行参数辨识 ===
共 34 组真实数据

辨识得出参数:   b1 = 0.3626 kg·m,  b2 = 0.1971 kg·m
理论标称参数:   b1 = 0.7314 kg·m,  b2 = 0.5100 kg·m
回归 R² 得分:   0.975340
参数绝对误差:   Δb1 = 0.3688,  Δb2 = 0.3129


In [3]:
print("\n\n=== 部署到控制器的前馈补偿公式 (C++式) ===")
# 在实际的高频 C++ 控制代码中，你不需要把所有矩阵重新展开。
# 既然 G = Y * [b1, b2]^T，我们只需要把 Y(q, g_vec) * theta 的解析解导出即可：
tau_g_sym = Y_sym * sp.Matrix([b1, b2])

print("\n// 控制器中用于重力补偿的计算代码：")
print(f"// 参数需在类初始化时装载: b1 = {theta_hat[0]:.6f}, b2 = {theta_hat[1]:.6f}")
print("double tau_g[3];")
for i in range(3):
    # 将符号转为 C 代码
    c_code = sp.ccode(sp.simplify(tau_g_sym[i]), assign_to=f"tau_g[{i}]")
    print(c_code)




=== 部署到控制器的前馈补偿公式 (C++式) ===

// 控制器中用于重力补偿的计算代码：
// 参数需在类初始化时装载: b1 = 0.318310, b2 = 0.186486
double tau_g[3];
tau_g[0] = (b1*cos(q2) + b2*cos(q2 + q3))*(gx*sin(q1) - gy*cos(q1));
tau_g[1] = b1*(gx*sin(q2)*cos(q1) + gy*sin(q1)*sin(q2) - gz*cos(q2)) + b2*(gx*sin(q2 + q3)*cos(q1) + gy*sin(q1)*sin(q2 + q3) - gz*cos(q2 + q3));
tau_g[2] = b2*(gx*sin(q2 + q3)*cos(q1) + gy*sin(q1)*sin(q2 + q3) - gz*cos(q2 + q3));


In [4]:

print("=== 从组合参数 b1, b2 反解物理参数 ===")
# 回归只能辨识组合参数：
#   b1 = m1*d1 + m2*L1
#   b2 = m2*d2
#
# 要拆出单独的物理量，需要通过独立测量（称重/CAD）已知部分参数作为约束。
# 典型策略：已知质量 m1, m2 和连杆长度 L1 → 反解质心距 d1, d2

# ---- 已知（由 CAD 读取或直接称重量） ----
m1_known = 1.512   # kg，连杆1质量（对应原 m2_old）
m2_known = 1.7     # kg，连杆2质量（对应原 m3_old）
L1_known = 0.29781 # m，连杆1长度

# ---- 用辨识出的 b1, b2 反解质心距 ----
b1_hat, b2_hat = theta_hat[0], theta_hat[1]

d2_hat = b2_hat / m2_known
d1_hat = (b1_hat - m2_known * L1_known) / m1_known

# ---- 标称参考值 ----
d1_nominal = l2_old / 2   # 0.14891 m
d2_nominal = r3_len        # 0.3 m

print(f"\n辨识物理参数（已知 m1={m1_known} kg, m2={m2_known} kg, L1={L1_known} m）：")
print(f"  d1 (连杆1质心距) = {d1_hat:.4f} m   (标称: {d1_nominal:.4f} m,  误差: {abs(d1_hat-d1_nominal)*1000:.2f} mm)")
print(f"  d2 (连杆2质心距) = {d2_hat:.4f} m   (标称: {d2_nominal:.4f} m,  误差: {abs(d2_hat-d2_nominal)*1000:.2f} mm)")

print("\n注意：")
print("  · d1、d2 的精度依赖于 m1、m2 的测量精度；")
print("  · 若质量未知，可同时采集不同构型下的多组数据，将 m1*d1、m2*L1 分别设为独立参数（扩展参数向量）。")
print("  · 若想完全解耦 m 与 d，需要在不同重力方向下采集数据（如倾斜平台实验）。")


=== 从组合参数 b1, b2 反解物理参数 ===

辨识物理参数（已知 m1=1.512 kg, m2=1.7 kg, L1=0.29781 m）：
  d1 (连杆1质心距) = -0.1243 m   (标称: 0.1489 m,  误差: 273.22 mm)
  d2 (连杆2质心距) = 0.1097 m   (标称: 0.3000 m,  误差: 190.30 mm)

注意：
  · d1、d2 的精度依赖于 m1、m2 的测量精度；
  · 若质量未知，可同时采集不同构型下的多组数据，将 m1*d1、m2*L1 分别设为独立参数（扩展参数向量）。
  · 若想完全解耦 m 与 d，需要在不同重力方向下采集数据（如倾斜平台实验）。
